###Task 4.1: One Validated Call

In [13]:

import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import Literal
from getpass import getpass

notes_df = pd.read_csv("/content/followup_notes.csv")

api_key = getpass("Enter your Gemini API key: ")
client = genai.Client(api_key=api_key)

def create_prompt(note_text):

    prompt = f"""
    CONTEXT:
    A healthcare clinic is analysing patient call-centre
    notes to understand why patients missed appointments.

    ROLE:
    You are an experienced healthcare data analyst.

    ACTION:
    Analyse the patient's note and identify the primary
    reason for missing the appointment.

    Choose exactly one root cause from the allowed
    categories defined in the response schema.

    Consider the actual meaning of the note, including
    informal language, abbreviations and Hinglish.

    If the reason is unclear, choose other and assign
    an appropriately low confidence score.

    FORMAT:
    Follow the supplied structured JSON response schema.
    Do not add markdown or additional commentary.

    TONE:
    Be objective, concise and evidence-based.
    Do not invent information that is absent from the note.

    PATIENT NOTE:
    {note_text}
    """

    return prompt


def analyse_note(note_text):

    prompt = create_prompt(note_text)

    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=FollowupInsight,
            thinking_config=types.ThinkingConfig(
                thinking_level="low"
            )
        )
    )

    raw_response = response.text

    if not raw_response:
        raise ValueError("Gemini returned an empty response")

    validated_result = FollowupInsight.model_validate_json(
        raw_response
    )

    return validated_result, raw_response


note_n001 = notes_df.loc[
    notes_df["note_id"] == "N001",
    "note_text"
].iloc[0]

result, raw_response = analyse_note(note_n001)

print("Validated Output:")
print(result.model_dump())

print("\nOutput Keys:")
print(list(result.model_dump().keys()))

print("\nRaw Response:")
print(raw_response)


Enter your Gemini API key: ··········
Validated Output:
{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.95, 'evidence': 'Patient said she never got any reminder message and completely forgot about the visit.', 'recommended_action': 'Send automated SMS reminders prior to appointments and reschedule for next week.'}

Output Keys:
['root_cause', 'confidence', 'evidence', 'recommended_action']

Raw Response:
{"root_cause":"forgot_or_no_reminder","confidence":0.95,"evidence":"Patient said she never got any reminder message and completely forgot about the visit.","recommended_action":"Send automated SMS reminders prior to appointments and reschedule for next week."}


###Task 4.2: Validate and Process the 10 Notes

In [21]:

import pandas as pd
import time

notes_df = pd.read_csv("/content/followup_notes.csv")

first_10_notes = notes_df.head(10)

analysed_results = []

for index, row in first_10_notes.iterrows():

    note_id = row["note_id"]
    appointment_id = row["appointment_id"]
    note_text = row["note_text"]

    result_row = {
        "note_id": note_id,
        "appointment_id": appointment_id,
        "note_text": note_text,
        "root_cause": None,
        "confidence": None,
        "evidence": None,
        "recommended_action": None,
        "status": None,
        "raw_response": None,
        "error": None
    }

    if pd.isna(note_text) or str(note_text).strip() == "":

        result_row["status"] = "skipped_blank"
        print(f"{note_id}: Blank note skipped")

    else:

        try:

            result, raw_response = analyse_note(str(note_text))

            result_row["root_cause"] = result.root_cause
            result_row["confidence"] = result.confidence
            result_row["evidence"] = result.evidence
            result_row["recommended_action"] = result.recommended_action
            result_row["status"] = "analysed"
            result_row["raw_response"] = raw_response

            print(f"{note_id}: Successfully analysed")

        except Exception as e:

            result_row["status"] = "error"
            result_row["error"] = str(e)

            print(f"{note_id}: Error - {e}")

        finally:
            time.sleep(1)

    analysed_results.append(result_row)

analysed_df = pd.DataFrame(analysed_results)

analysed_df.to_csv(
    "/content/notes_analysed.csv",
    index=False
)

print("\nBatch processing completed")
print("\nStatus Summary:")
print(analysed_df["status"].value_counts())

print("\nAnalysed Data:")
display(analysed_df)

print("\nCSV file saved successfully")


N001: Successfully analysed
N002: Successfully analysed
N003: Successfully analysed
N004: Blank note skipped
N005: Successfully analysed
N006: Successfully analysed
N007: Successfully analysed
N008: Successfully analysed
N009: Successfully analysed
N010: Successfully analysed

Batch processing completed

Status Summary:
status
analysed         9
skipped_blank    1
Name: count, dtype: int64

Analysed Data:


,note_id,appointment_id,note_text,root_cause,confidence,evidence,recommended_action,status,raw_response,error
0,N001,A0872,Patient said she never got any reminder messag...,forgot_or_no_reminder,0.98,Patient said she never got any reminder messag...,Send automated SMS and email reminders prior t...,analysed,"{""root_cause"":""forgot_or_no_reminder"",""confide...",None
1,N002,A0897,Lives far from the clinic and could not find a...,distance_or_transport,0.98,Lives far from the clinic and could not find a...,Offer telehealth options or help coordinate tr...,analysed,"{\n ""root_cause"": ""distance_or_transport"",\n ...",None
2,N003,A0526,Patient felt the consultation fee was too high...,cost_or_fee,0.98,Patient felt the consultation fee was too high...,Provide information regarding financial assist...,analysed,"{""root_cause"":""cost_or_fee"",""confidence"":0.98,...",None
3,N004,A0926,NaN,None,NaN,None,None,skipped_blank,None,None
4,N005,A0534,"Had high fever and body ache on the day, staye...",illness_or_emergency,0.95,"Had high fever and body ache on the day, staye...",Follow up with the patient to reschedule their...,analysed,"{""root_cause"":""illness_or_emergency"",""confiden...",None
5,N006,A0507,Manager called an urgent meeting at the office...,work_or_schedule,0.95,Manager called an urgent meeting at the office...,Offer evening or weekend appointment slots to ...,analysed,"{""root_cause"":""work_or_schedule"",""confidence"":...",None
6,N007,A0333,"Bhool gaya, SMS nahi aaya tha. Dobara booking ...",forgot_or_no_reminder,0.95,"Bhool gaya, SMS nahi aaya tha.",Reschedule the appointment and verify SMS remi...,analysed,"{\n ""root_cause"": ""forgot_or_no_reminder"",\n ...",None
7,N008,A0222,"City bus strike that day, no auto available ne...",distance_or_transport,0.95,"City bus strike that day, no auto available ne...",Offer telehealth options or provide assistance...,analysed,"{""root_cause"":""distance_or_transport"",""confide...",None
8,N009,A0977,"Phone was switched off, could not reach the pa...",other,0.30,"Phone was switched off, could not reach the pa...",Try reaching out via alternative communication...,analysed,"{\n ""root_cause"": ""other"",\n ""confidence"": 0...",None
9,N010,A0596,Had a shift change at the factory and could no...,work_or_schedule,0.95,Had a shift change at the factory and could no...,Reschedule the patient for a Saturday appointm...,analysed,"{\n ""root_cause"": ""work_or_schedule"",\n ""con...",None



CSV file saved successfully


###Task 4.3: Break It on Purpose

In [24]:
import time
import pandas as pd
from pydantic import ValidationError
from google.genai import types, errors

notes_df = pd.read_csv("/content/followup_notes.csv")

print("PART 1: Pydantic validation testing")

valid_sample = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 0.9,
    "evidence": "The patient said they forgot the appointment.",
    "recommended_action": "Send an SMS reminder before the appointment.",
}

invalid_confidence = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 1.5,
    "evidence": "The patient said they forgot the appointment.",
    "recommended_action": "Send an SMS reminder before the appointment.",
}

missing_field = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 0.9,
    "evidence": "The patient said they forgot the appointment.",
}

validated = FollowupInsight.model_validate(valid_sample)
print("\nValid data:")
print(validated.model_dump())

try:
    FollowupInsight.model_validate(invalid_confidence)
except ValidationError as e:
    print("\nInvalid confidence error:")
    print(e)

try:
    FollowupInsight.model_validate(missing_field)
except ValidationError as e:
    print("\nMissing field error:")
    print(e)


print("\nPART 2: V1 vs V2 prompt comparison")

api_call_count = 0
MODELS_TO_TRY = ["gemini-2.5-flash", "gemini-2.5-flash-lite"]


def call_gemini(prompt, retries=2):
    global api_call_count

    for model_name in MODELS_TO_TRY:
        for attempt in range(retries + 1):
            api_call_count += 1
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        response_mime_type="application/json",
                        response_schema=FollowupInsight,
                        thinking_config=types.ThinkingConfig(thinking_level="low"),
                    ),
                )
                raw_response = response.text
                if not raw_response:
                    raise ValueError("Gemini returned an empty response")
                result = FollowupInsight.model_validate_json(raw_response)
                return result, raw_response
            except errors.APIError as e:
                if e.code == 503 and attempt < retries:
                    time.sleep(2 ** attempt)
                    continue
                break
            except Exception:
                if attempt == retries:
                    raise
                time.sleep(1)

    raise RuntimeError("All models failed during call_gemini")


def create_v1_prompt(note_text):
    return f"""
Analyse the following patient follow-up note.
Identify the main reason for the missed appointment.

Return the result in JSON with these fields:
root_cause, confidence, evidence, recommended_action.

Use only these root_cause values:
forgot_or_no_reminder,
distance_or_transport,
cost_or_fee,
illness_or_emergency,
work_or_schedule,
other.

Patient note:
{note_text}
"""


comparison_results = []

for note_id in ["N007", "N009"]:
    note_row = notes_df.loc[notes_df["note_id"] == note_id]
    if note_row.empty:
        print(f"{note_id} not found in the dataset.")
        continue

    note_text = note_row.iloc[0]["note_text"]
    if pd.isna(note_text) or str(note_text).strip() == "":
        print(f"{note_id} contains a blank note.")
        continue

    print(f"\nAnalysing {note_id}...")
    row_result = {"note_id": note_id, "note_text": note_text}

    try:
        v1_result, v1_raw = call_gemini(create_v1_prompt(str(note_text)))
        row_result["V1_root_cause"] = v1_result.root_cause
        row_result["V1_confidence"] = v1_result.confidence
        row_result["V1_evidence"] = v1_result.evidence
        row_result["V1_recommended_action"] = v1_result.recommended_action
        row_result["V1_raw_response"] = v1_raw
    except Exception as e:
        row_result["V1_error"] = str(e)
        print(f"V1 error: {e}")

    try:
        v2_result, v2_raw = call_gemini(create_prompt(str(note_text)))
        row_result["V2_root_cause"] = v2_result.root_cause
        row_result["V2_confidence"] = v2_result.confidence
        row_result["V2_evidence"] = v2_result.evidence
        row_result["V2_recommended_action"] = v2_result.recommended_action
        row_result["V2_raw_response"] = v2_raw
    except Exception as e:
        row_result["V2_error"] = str(e)
        print(f"V2 error: {e}")

    comparison_results.append(row_result)

comparison_df = pd.DataFrame(comparison_results)
print("\nV1 vs V2 Comparison:")
display(comparison_df)

comparison_df.to_csv("/content/prompt_comparison.csv", index=False)
print("\nTotal Gemini API calls:", api_call_count)
print("API call requirement (<25):", api_call_count < 25)
print("Comparison saved as prompt_comparison.csv")

PART 1: Pydantic validation testing

Valid data:
{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.9, 'evidence': 'The patient said they forgot the appointment.', 'recommended_action': 'Send an SMS reminder before the appointment.'}

Invalid confidence error:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.5, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Missing field error:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'forgot_or...orgot the appointment.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing

PART 2: V1 vs V2 prompt comparison

Analysing N007...
V1 error: All models failed during call_gemini
V2 error: All models failed during call_gemini

Analysing N009...
V1 error: All models failed during call_gemini
V2 error: Al

,note_id,note_text,V1_error,V2_error
0,N007,"Bhool gaya, SMS nahi aaya tha. Dobara booking ...",All models failed during call_gemini,All models failed during call_gemini
1,N009,"Phone was switched off, could not reach the pa...",All models failed during call_gemini,All models failed during call_gemini



Total Gemini API calls: 8
API call requirement (<25): True
Comparison saved as prompt_comparison.csv


###Task 4.4: Recommendation & GitHub Submission

In [26]:
import pandas as pd

# 1. Load analysed notes and cleaned appointments
notes_df = pd.read_csv("/content/notes_analysed.csv")
appt_df = pd.read_csv("/content/appointments_clean.csv")

# Keep only rows that were successfully analysed
valid_notes = notes_df[notes_df["status"] == "analysed"].copy()

# 2. Merge on appointment_id (adjust key name if your column is named differently)
merged_df = pd.merge(
    valid_notes,
    appt_df,
    on="appointment_id",
    how="left",
)

# 3. Cross-tabulate root_cause against reminder_sent
ct_counts = pd.crosstab(
    merged_df["root_cause"],
    merged_df["reminder_sent"],
    margins=True,
    dropna=False,
)
print("Cross-Tabulation (Counts):")
print(ct_counts)

# Optional: row-wise percentages to see reminder breakdown within each root cause
ct_pct = pd.crosstab(
    merged_df["root_cause"],
    merged_df["reminder_sent"],
    normalize="index",
).round(3)
print("\nCross-Tabulation (Row Proportions):")
print(ct_pct)

# Save merged table for reference
merged_df.to_csv("/content/merged_notes_appointments.csv", index=False)

Cross-Tabulation (Counts):
reminder_sent          No  Yes  NaN  All
root_cause                              
cost_or_fee             0    1    0    1
distance_or_transport   0    1    1    2
forgot_or_no_reminder   2    0    0    2
illness_or_emergency    0    1    0    1
other                   0    1    0    1
work_or_schedule        1    1    0    2
All                     3    5    0    9

Cross-Tabulation (Row Proportions):
reminder_sent           No  Yes
root_cause                     
cost_or_fee            0.0  1.0
distance_or_transport  0.0  1.0
forgot_or_no_reminder  1.0  0.0
illness_or_emergency   0.0  1.0
other                  0.0  1.0
work_or_schedule       0.5  0.5
